# 8/10 이후

- 변수 조합 1차 비교 및 후보 선정


4개 알고리즘별로 동일한 데이터 분할과 기본 학습 조건을 적용하여 5개 변수 조합을 비교하고, PR-AUC를 중심으로 성능이 우수한 조합 1~3개를 후보로 선정합니다. 알고리즘마다 변수 조합의 효과가 다를 수 있으므로 선정 결과는 달라도 되며, 조합 간 성능 차이가 미미하면 임의로 제외하지 않고 시간순 교차검증과 하이퍼파라미터 튜닝 단계에서 추가로 비교합니다.

In [3]:
## 라이브러리 불러오기

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

RANDOM_STATE = 42
TARGET = "is_fraud"
THRESHOLD = 0.9

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.7.0


In [4]:
## 데이터 불러오기

DATA_PATH = r"C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\data\fraud_full_features.csv"

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["trans_date_trans_time"]
)

print("데이터 크기:", df.shape)
display(df.head())

데이터 크기: (1296675, 31)


,trans_date_trans_time,cc_num,merchant,category,amt,is_fraud,recent_24h_high_amt_count,category_recent_fraud_rate,category_recent_fraud_rate_missing,count_30min,...,amt_to_prior_median_ratio,is_10x_prior_median,has_prior_normal_transaction,outside_trans_hours_80,is_online,risk_time_22_04,interact_repeat_category,merchant_change_count,rolling_sum_amt_1h,is_high_amt
0,2019-01-01 12:47:15,60416207185,"fraud_Jones, Sawayn and Romaguera",misc_net,7.27,0,0,0.000000,0,0,...,NaN,0,0,0,1,0,0.0,0,7.27,0
1,2019-01-02 08:44:57,60416207185,fraud_Berge LLC,gas_transport,52.94,0,0,0.006154,0,1,...,7.281981,0,1,0,0,0,0.0,1,52.94,0
2,2019-01-02 08:47:36,60416207185,fraud_Luettgen PLC,gas_transport,82.08,0,0,0.006135,0,2,...,2.726457,0,1,0,0,0,0.0,1,135.02,0
3,2019-01-02 12:38:14,60416207185,fraud_Daugherty LLC,kids_pets,34.79,0,0,0.000000,0,1,...,0.657159,0,1,0,0,0,0.0,1,34.79,0
4,2019-01-02 13:10:46,60416207185,fraud_Beier and Sons,home,27.18,0,0,0.000000,0,1,...,0.619628,0,1,0,0,0,0.0,1,61.97,0


In [5]:
## 모든 팀원이 공통으로 사용할 80:20 행 분할

# 전체 데이터의 행 위치
all_indices = np.arange(len(df))

# 전체 타깃
y_all = df[TARGET].astype(int)

# 이상거래 비율을 유지하는 층화 분할
train_idx, val_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_all
)

print("Train 크기:", f"{len(train_idx):,}")
print("Validation 크기:", f"{len(val_idx):,}")

Train 크기: 1,037,340
Validation 크기: 259,335


80:20 분할 잘 됨

In [6]:
## 분할이 제대로 되었는지 확인

y_train_common = y_all.iloc[train_idx]
y_val_common = y_all.iloc[val_idx]

print(
    f"Train 이상거래: {y_train_common.sum():,}건 "
    f"({y_train_common.mean():.4%})"
)

print(
    f"Validation 이상거래: {y_val_common.sum():,}건 "
    f"({y_val_common.mean():.4%})"
)

Train 이상거래: 6,005건 (0.5789%)
Validation 이상거래: 1,501건 (0.5788%)


train셋, val셋에서 이상거래율 비슷 OK

In [7]:
## train 데이터에서 scale_pos_weight(가중치) 계산

n_negative = (y_train_common == 0).sum()
n_positive = (y_train_common == 1).sum()

if n_positive == 0:
    raise ValueError(
        "Train 데이터에 이상거래가 없어 scale_pos_weight를 계산할 수 없습니다."
    )

scale_pos_weight = n_negative / n_positive

print(f"Train 정상거래 수: {n_negative:,}")
print(f"Train 이상거래 수: {n_positive:,}")
print(f"scale_pos_weight: {scale_pos_weight:.6f}")

Train 정상거래 수: 1,031,335
Train 이상거래 수: 6,005
scale_pos_weight: 171.746045


In [8]:
feature_combinations = {
    "조합1": [
        "category",
        "amt",
        "trans_hour",
        "age",
        "recent_24h_high_amt_count",
        "amt_to_prior_median_ratio",
        "rolling_sum_amt_1h",
        "amt_zscore_card",
        "prior_normal_median_amt",
        "count_30min"
    ],

    "조합2": [
        "category",
        "amt",
        "trans_hour",
        "age",
        "recent_24h_high_amt_count",
        "amt_to_prior_median_ratio",
        "rolling_sum_amt_1h",
        "amt_zscore_card",
        "prior_normal_median_amt",
        "count_30min",
        "merchant_change_count"
    ],

    "조합3": [
        "category",
        "amt",
        "trans_hour",
        "age",
        "recent_24h_high_amt_count",
        "amt_to_prior_median_ratio",
        "rolling_sum_amt_1h",
        "amt_zscore_card",
        "prior_normal_median_amt",
        "count_30min",
        "high_speed"
    ],

    "조합4": [
        "recent_24h_high_amt_count",
        "amt_to_prior_median_ratio",
        "category",
        "amt",
        "trans_hour",
        "age"
    ],

    "조합5": [
        "category",
        "amt",
        "is_online",
        "recent_24h_high_amt_count",
        "category_recent_fraud_rate",
        "speed_2",
        "customer_mean_amt",
        "customer_std_amt",
        "amt_ratio_to_mean",
        "amt_zscore_card",
        "customer_transaction_count",
        "trans_hour",
        "age",
        "rolling_sum_amt_1h",
        "prior_normal_median_amt",
        "amt_to_prior_median_ratio",
        "risk_time_22_04",
        "interact_repeat_category",
        "has_prior_normal_transaction"
    ]
}

In [9]:
for combination_name, feature_list in feature_combinations.items():
    print(f"{combination_name}: {len(feature_list)}개 변수")
    print(feature_list)
    print()

조합1: 10개 변수
['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count', 'amt_to_prior_median_ratio', 'rolling_sum_amt_1h', 'amt_zscore_card', 'prior_normal_median_amt', 'count_30min']

조합2: 11개 변수
['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count', 'amt_to_prior_median_ratio', 'rolling_sum_amt_1h', 'amt_zscore_card', 'prior_normal_median_amt', 'count_30min', 'merchant_change_count']

조합3: 11개 변수
['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count', 'amt_to_prior_median_ratio', 'rolling_sum_amt_1h', 'amt_zscore_card', 'prior_normal_median_amt', 'count_30min', 'high_speed']

조합4: 6개 변수
['recent_24h_high_amt_count', 'amt_to_prior_median_ratio', 'category', 'amt', 'trans_hour', 'age']

조합5: 19개 변수
['category', 'amt', 'is_online', 'recent_24h_high_amt_count', 'category_recent_fraud_rate', 'speed_2', 'customer_mean_amt', 'customer_std_amt', 'amt_ratio_to_mean', 'amt_zscore_card', 'customer_transaction_count', 'trans_hour', 'age', 'rolling_sum_amt_1

In [10]:
## csv에 누락된 변수가 있는지 확인

all_columns_exist = True

for combination_name, feature_list in feature_combinations.items():
    missing_columns = [
        column
        for column in feature_list
        if column not in df.columns
    ]

    if missing_columns:
        all_columns_exist = False
        print(f"❌ {combination_name} 누락 변수: {missing_columns}")
    else:
        print(f"✅ {combination_name}: 모든 변수 확인 완료")

if all_columns_exist:
    print("\n5개 조합에 필요한 변수가 모두 존재합니다.")
else:
    print("\n누락된 변수의 실제 열 이름을 확인해야 합니다.")

✅ 조합1: 모든 변수 확인 완료
✅ 조합2: 모든 변수 확인 완료
✅ 조합3: 모든 변수 확인 완료
✅ 조합4: 모든 변수 확인 완료
✅ 조합5: 모든 변수 확인 완료

5개 조합에 필요한 변수가 모두 존재합니다.


In [11]:
# 5개 조합에서 사용하는 전체 변수 목록
all_features = list(dict.fromkeys(
    feature
    for feature_list in feature_combinations.values()
    for feature in feature_list
))

print(f"중복 제외 전체 변수 수: {len(all_features)}개")

중복 제외 전체 변수 수: 22개


In [12]:
%pip install jinja2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
feature_check_df = pd.DataFrame({
    "dtype": df[all_features].dtypes.astype(str),
    "결측치수": df[all_features].isna().sum(),
    "결측치비율_pct": (
        df[all_features].isna().mean() * 100
    ),
    "고유값수": df[all_features].nunique(dropna=False)
}).sort_values(
    by="결측치수",
    ascending=False
)

display(
    feature_check_df.style.format({
        "결측치비율_pct": "{:.4f}"
    })
)

,dtype,결측치수,결측치비율_pct,고유값수
amt_to_prior_median_ratio,float64,1649,0.1272,1247936
prior_normal_median_amt,float64,1649,0.1272,15827
amt,float64,0,0.0000,52928
trans_hour,int64,0,0.0000,24
age,int64,0,0.0000,83
category,str,0,0.0000,14
recent_24h_high_amt_count,int64,0,0.0000,10
rolling_sum_amt_1h,float64,0,0.0000,69038
amt_zscore_card,float64,0,0.0000,1294710
count_30min,int64,0,0.0000,6


실무적으로 각 모델의 최대 성능을 비교하려는 것이기 때문에,

category(현재 혼자 str 타입)을 원-핫 인코딩이 아닌, LightGBM 자체 범주형 처리로.

In [15]:
## lightGBM 자체 범주형 처리 (category 변수만)

def prepare_combination_data(
    df,
    feature_list,
    train_idx,
    val_idx,
    target="is_fraud"
):
    # 선택한 변수와 타깃 복사
    X_all = df[feature_list].copy()
    y_all = df[target].astype(int).copy()

    # LightGBM이 자체 처리할 범주형 변수
    categorical_features = []

    if "category" in X_all.columns:
        X_all["category"] = X_all["category"].astype("category")
        categorical_features.append("category")

    # 모든 조합에 동일한 행 분할 적용
    X_train = X_all.iloc[train_idx].copy()
    X_val = X_all.iloc[val_idx].copy()

    y_train = y_all.iloc[train_idx].copy()
    y_val = y_all.iloc[val_idx].copy()

    return {
        "X_train": X_train,
        "X_val": X_val,
        "y_train": y_train,
        "y_val": y_val,
        "categorical_features": categorical_features
    }

In [16]:
combination1_data = prepare_combination_data(
    df=df,
    feature_list=feature_combinations["조합1"],
    train_idx=train_idx,
    val_idx=val_idx,
    target=TARGET
)

print("Train 크기:", combination1_data["X_train"].shape)
print("Validation 크기:", combination1_data["X_val"].shape)

print("\nTrain 자료형:")
print(combination1_data["X_train"].dtypes)

print(
    "\n범주형 변수:",
    combination1_data["categorical_features"]
)

print(
    "\nTrain 결측치 수:",
    combination1_data["X_train"].isna().sum().sum()
)

print(
    "Validation 결측치 수:",
    combination1_data["X_val"].isna().sum().sum()
)

print(
    "\nTrain 이상거래율:",
    combination1_data["y_train"].mean()
)

print(
    "Validation 이상거래율:",
    combination1_data["y_val"].mean()
)

Train 크기: (1037340, 10)
Validation 크기: (259335, 10)

Train 자료형:
category                     category
amt                           float64
trans_hour                      int64
age                             int64
recent_24h_high_amt_count       int64
amt_to_prior_median_ratio     float64
rolling_sum_amt_1h            float64
amt_zscore_card               float64
prior_normal_median_amt       float64
count_30min                     int64
dtype: object

범주형 변수: ['category']

Train 결측치 수: 2636
Validation 결측치 수: 662

Train 이상거래율: 0.00578884454470087
Validation 이상거래율: 0.005787880540613492


결측치 수는 prior_normal_median_amt, amt_to_prior_median_ratio의 NaN이기 때문에 ㄱㅊ음

In [19]:
from sklearn import set_config

set_config(display="text")

## 조합1 (민정 Model 40)

In [23]:
from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

model_combination1_v2 = LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_combination1_v2.fit(
    combination1_data["X_train"],
    combination1_data["y_train"],
    eval_set=[
        (
            combination1_data["X_val"],
            combination1_data["y_val"]
        )
    ],
    eval_metric="auc",
    categorical_feature=combination1_data["categorical_features"],
    callbacks=[
        early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
            verbose=True
        ),
        log_evaluation(period=100)
    ]
);

c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.998823
[200]	valid_0's auc: 0.999258
[300]	valid_0's auc: 0.999404
[400]	valid_0's auc: 0.99944
[500]	valid_0's auc: 0.999451
Early stopping, best iteration is:
[492]	valid_0's auc: 0.999456
Evaluated only: auc


In [ ]:
print("최적 반복 횟수:", model_combination1_v2.best_iteration_)
print("최적 점수:")
print(model_combination1_v2.best_score_)
print("평가 지표:")
print(model_combination1_v2.evals_result_.keys())
print(model_combination1_v2.evals_result_["valid_0"].keys())

최적 반복 횟수: 492
최적 점수:
defaultdict(<class 'collections.OrderedDict'>, {'valid_0': OrderedDict([('auc', np.float64(0.9994557772807843))])})
평가 지표:
dict_keys(['valid_0'])
odict_keys(['auc'])


In [ ]:
## 492개 트리를 사용한 예측확률로 임계값 0.90 성능을 계산
## 하기 전에,  예측확률을 변수에 저장만 하는 코드.

val_probability_combination1_v2 = (
    model_combination1_v2.predict_proba(
        combination1_data["X_val"],
        num_iteration=model_combination1_v2.best_iteration_
    )[:, 1]
)

In [ ]:
import numpy as np

print("예측확률 개수:", len(val_probability_combination1_v2))
print("Validation 행 수:", len(combination1_data["X_val"]))

print("최소 확률:", val_probability_combination1_v2.min())
print("최대 확률:", val_probability_combination1_v2.max())
print("평균 확률:", val_probability_combination1_v2.mean())

print("\n확률 분위수:")
print(
    np.quantile(
        val_probability_combination1_v2,
        [0, 0.5, 0.9, 0.99, 0.995, 0.999, 1.0]
    )
)

예측확률 개수: 259335
Validation 행 수: 259335
최소 확률: 4.661481550417692e-09
최대 확률: 0.9999905373099288
평균 확률: 0.0114293834569628

확률 분위수:
[4.66148155e-09 7.87283149e-05 2.62484883e-03 4.09374977e-01
 9.92786506e-01 9.99934335e-01 9.99990537e-01]


In [28]:
## 공통 임계값 0.90 적용해서  성능 확인.

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

COMMON_THRESHOLD = 0.90

y_val_combination1 = combination1_data["y_val"].to_numpy()

val_prediction_combination1_v2 = (
    val_probability_combination1_v2 >= COMMON_THRESHOLD
).astype(int)

precision_v2 = precision_score(
    y_val_combination1,
    val_prediction_combination1_v2,
    zero_division=0
)

recall_v2 = recall_score(
    y_val_combination1,
    val_prediction_combination1_v2,
    zero_division=0
)

f1_v2 = f1_score(
    y_val_combination1,
    val_prediction_combination1_v2,
    zero_division=0
)

roc_auc_v2 = roc_auc_score(
    y_val_combination1,
    val_probability_combination1_v2
)

pr_auc_v2 = average_precision_score(
    y_val_combination1,
    val_probability_combination1_v2
)

tn, fp, fn, tp = confusion_matrix(
    y_val_combination1,
    val_prediction_combination1_v2,
    labels=[0, 1]
).ravel()

print(f"최적 반복 횟수: {model_combination1_v2.best_iteration_}")
print(f"공통 임계값: {COMMON_THRESHOLD:.2f}")
print(f"Precision: {precision_v2:.6f}")
print(f"Recall: {recall_v2:.6f}")
print(f"F1-score: {f1_v2:.6f}")
print(f"ROC-AUC: {roc_auc_v2:.6f}")
print(f"PR-AUC: {pr_auc_v2:.6f}")

print("\nConfusion Matrix")
print(f"TN: {tn:,}")
print(f"FP: {fp:,}")
print(f"FN: {fn:,}")
print(f"TP: {tp:,}")

print("\n예측 결과")
print("실제 이상거래 수:", int(y_val_combination1.sum()))
print("예측 이상거래 수:", int(val_prediction_combination1_v2.sum()))

최적 반복 횟수: 492
공통 임계값: 0.90
Precision: 0.899685
Recall: 0.950033
F1-score: 0.924174
ROC-AUC: 0.999456
PR-AUC: 0.972656

Confusion Matrix
TN: 257,675
FP: 159
FN: 75
TP: 1,426

예측 결과
실제 이상거래 수: 1501
예측 이상거래 수: 1585


- 실제 이상거래 1,501건 중 1,426건을 탐지.
- 이상거래를 75건 놓쳤고, 
- 정상거래 159건을 이상거래로 오탐.

##### 다시 한 번 짚고 넘어가는 'lightGBM 기본 설정'
- 동일한 Train/Validation 인덱스
- n_estimators=3000
- learning_rate=0.03
- 동일한 하이퍼파라미터와 scale_pos_weight
- eval_metric="auc"
- early_stopping=100
- first_metric_only=True
- 공통 임계값 0.90
- category는 LightGBM 자체 범주형 처리

## 조합 2 (민정 모델 42)

In [29]:
## 조합2 데이터 준비

combination2_data = prepare_combination_data(
    df=df,
    feature_list=feature_combinations["조합2"],
    train_idx=train_idx,
    val_idx=val_idx,
    target=TARGET
)

print("조합2 변수:")
print(feature_combinations["조합2"])

print("\nTrain 크기:", combination2_data["X_train"].shape)
print("Validation 크기:", combination2_data["X_val"].shape)

print("\n자료형:")
print(combination2_data["X_train"].dtypes)

print(
    "\n범주형 변수:",
    combination2_data["categorical_features"]
)

print(
    "Train/Validation 행 수 일치:",
    len(combination2_data["X_train"]) == len(combination1_data["X_train"])
    and len(combination2_data["X_val"]) == len(combination1_data["X_val"])
)

조합2 변수:
['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count', 'amt_to_prior_median_ratio', 'rolling_sum_amt_1h', 'amt_zscore_card', 'prior_normal_median_amt', 'count_30min', 'merchant_change_count']

Train 크기: (1037340, 11)
Validation 크기: (259335, 11)

자료형:
category                     category
amt                           float64
trans_hour                      int64
age                             int64
recent_24h_high_amt_count       int64
amt_to_prior_median_ratio     float64
rolling_sum_amt_1h            float64
amt_zscore_card               float64
prior_normal_median_amt       float64
count_30min                     int64
merchant_change_count           int64
dtype: object

범주형 변수: ['category']
Train/Validation 행 수 일치: True


In [30]:
## 학습
# 
from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

model_combination2 = LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_combination2.fit(
    combination2_data["X_train"],
    combination2_data["y_train"],
    eval_set=[
        (
            combination2_data["X_val"],
            combination2_data["y_val"]
        )
    ],
    eval_metric="auc",
    categorical_feature=combination2_data["categorical_features"],
    callbacks=[
        early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
            verbose=True
        ),
        log_evaluation(period=100)
    ]
);

c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.998823
[200]	valid_0's auc: 0.999258
[300]	valid_0's auc: 0.999379
[400]	valid_0's auc: 0.999413
[500]	valid_0's auc: 0.999397
Early stopping, best iteration is:
[412]	valid_0's auc: 0.999419
Evaluated only: auc


In [31]:
print("최적 반복 횟수:", model_combination2.best_iteration_)
print("최적 Validation ROC-AUC:")
print(model_combination2.best_score_["valid_0"]["auc"])

최적 반복 횟수: 412
최적 Validation ROC-AUC:
0.9994186618489438


In [32]:
## validation 예측확률 생성

import numpy as np

val_probability_combination2 = model_combination2.predict_proba(
    combination2_data["X_val"],
    num_iteration=model_combination2.best_iteration_
)[:, 1]

print("예측확률 개수:", len(val_probability_combination2))
print("Validation 행 수:", len(combination2_data["X_val"]))
print("최소 확률:", val_probability_combination2.min())
print("최대 확률:", val_probability_combination2.max())
print("평균 확률:", val_probability_combination2.mean())

print("\n확률 분위수:")
print(
    np.quantile(
        val_probability_combination2,
        [0, 0.5, 0.9, 0.99, 0.995, 0.999, 1.0]
    )
)

예측확률 개수: 259335
Validation 행 수: 259335
최소 확률: 3.064396598151915e-08
최대 확률: 0.9999800962378019
평균 확률: 0.012361333401292515

확률 분위수:
[3.06439660e-08 1.22323898e-04 3.48835373e-03 4.67683051e-01
 9.92596283e-01 9.99887321e-01 9.99980096e-01]


In [33]:
## 공통 임계값 0.90으로 평가

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

COMMON_THRESHOLD = 0.90

y_val_combination2 = combination2_data["y_val"].to_numpy()

val_prediction_combination2 = (
    val_probability_combination2 >= COMMON_THRESHOLD
).astype(int)

precision_combination2 = precision_score(
    y_val_combination2,
    val_prediction_combination2,
    zero_division=0
)

recall_combination2 = recall_score(
    y_val_combination2,
    val_prediction_combination2,
    zero_division=0
)

f1_combination2 = f1_score(
    y_val_combination2,
    val_prediction_combination2,
    zero_division=0
)

roc_auc_combination2 = roc_auc_score(
    y_val_combination2,
    val_probability_combination2
)

pr_auc_combination2 = average_precision_score(
    y_val_combination2,
    val_probability_combination2
)

tn2, fp2, fn2, tp2 = confusion_matrix(
    y_val_combination2,
    val_prediction_combination2,
    labels=[0, 1]
).ravel()

print(f"최적 반복 횟수: {model_combination2.best_iteration_}")
print(f"공통 임계값: {COMMON_THRESHOLD:.2f}")
print(f"Precision: {precision_combination2:.6f}")
print(f"Recall: {recall_combination2:.6f}")
print(f"F1-score: {f1_combination2:.6f}")
print(f"ROC-AUC: {roc_auc_combination2:.6f}")
print(f"PR-AUC: {pr_auc_combination2:.6f}")

print("\nConfusion Matrix")
print(f"TN: {tn2:,}")
print(f"FP: {fp2:,}")
print(f"FN: {fn2:,}")
print(f"TP: {tp2:,}")

print("\n예측 결과")
print("실제 이상거래 수:", int(y_val_combination2.sum()))
print("예측 이상거래 수:", int(val_prediction_combination2.sum()))

최적 반복 횟수: 412
공통 임계값: 0.90
Precision: 0.884758
Recall: 0.951366
F1-score: 0.916854
ROC-AUC: 0.999419
PR-AUC: 0.971890

Confusion Matrix
TN: 257,648
FP: 186
FN: 73
TP: 1,428

예측 결과
실제 이상거래 수: 1501
예측 이상거래 수: 1614


조합2는 조합1보다 이상거래를 2건 더 탐지했지만, 정상거래를 이상거래로 잘못 잡은 건수가 27건 증가함. 

그 결과 Precision과 F1-score가 낮아짐.

현재 순위
1. 조합1 (F1 : 0.924174)
2. 조합2 (F1 : 0.916854)

### 조합 3 (민정 모델 44)

In [34]:
## 조합3 데이터 준비

combination3_data = prepare_combination_data(
    df=df,
    feature_list=feature_combinations["조합3"],
    train_idx=train_idx,
    val_idx=val_idx,
    target=TARGET
)

print("조합3 변수:")
print(feature_combinations["조합3"])

print("\nTrain 크기:", combination3_data["X_train"].shape)
print("Validation 크기:", combination3_data["X_val"].shape)
print("\n범주형 변수:", combination3_data["categorical_features"])

print(
    "조합1과 행 수 일치:",
    len(combination3_data["X_train"]) == len(combination1_data["X_train"])
    and len(combination3_data["X_val"]) == len(combination1_data["X_val"])
)

조합3 변수:
['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count', 'amt_to_prior_median_ratio', 'rolling_sum_amt_1h', 'amt_zscore_card', 'prior_normal_median_amt', 'count_30min', 'high_speed']

Train 크기: (1037340, 11)
Validation 크기: (259335, 11)

범주형 변수: ['category']
조합1과 행 수 일치: True


In [35]:
## 학습

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

model_combination3 = LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_combination3.fit(
    combination3_data["X_train"],
    combination3_data["y_train"],
    eval_set=[
        (
            combination3_data["X_val"],
            combination3_data["y_val"]
        )
    ],
    eval_metric="auc",
    categorical_feature=combination3_data["categorical_features"],
    callbacks=[
        early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
            verbose=True
        ),
        log_evaluation(period=100)
    ]
);

print("최적 반복 횟수:", model_combination3.best_iteration_)
print(
    "최적 Validation ROC-AUC:",
    model_combination3.best_score_["valid_0"]["auc"]
)

c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.998815
[200]	valid_0's auc: 0.999292
[300]	valid_0's auc: 0.999422
[400]	valid_0's auc: 0.999452
Early stopping, best iteration is:
[381]	valid_0's auc: 0.999456
Evaluated only: auc
최적 반복 횟수: 381
최적 Validation ROC-AUC: 0.9994561687447165


In [36]:
## validation 예측확률 생성

import numpy as np

val_probability_combination3 = model_combination3.predict_proba(
    combination3_data["X_val"],
    num_iteration=model_combination3.best_iteration_
)[:, 1]

print("예측확률 개수:", len(val_probability_combination3))
print("Validation 행 수:", len(combination3_data["X_val"]))
print("최소 확률:", val_probability_combination3.min())
print("최대 확률:", val_probability_combination3.max())
print("평균 확률:", val_probability_combination3.mean())

print("\n확률 분위수:")
print(
    np.quantile(
        val_probability_combination3,
        [0, 0.5, 0.9, 0.99, 0.995, 0.999, 1.0]
    )
)

예측확률 개수: 259335
Validation 행 수: 259335
최소 확률: 1.1469886671787456e-08
최대 확률: 0.9999572619584892
평균 확률: 0.01267587553611143

확률 분위수:
[1.14698867e-08 1.48775063e-04 3.81707442e-03 4.78184645e-01
 9.92843992e-01 9.99852965e-01 9.99957262e-01]


In [37]:
## 공통 임계값 0.90으로 평가

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

COMMON_THRESHOLD = 0.90

y_val_combination3 = combination3_data["y_val"].to_numpy()

val_prediction_combination3 = (
    val_probability_combination3 >= COMMON_THRESHOLD
).astype(int)

precision_combination3 = precision_score(
    y_val_combination3,
    val_prediction_combination3,
    zero_division=0
)

recall_combination3 = recall_score(
    y_val_combination3,
    val_prediction_combination3,
    zero_division=0
)

f1_combination3 = f1_score(
    y_val_combination3,
    val_prediction_combination3,
    zero_division=0
)

roc_auc_combination3 = roc_auc_score(
    y_val_combination3,
    val_probability_combination3
)

pr_auc_combination3 = average_precision_score(
    y_val_combination3,
    val_probability_combination3
)

tn3, fp3, fn3, tp3 = confusion_matrix(
    y_val_combination3,
    val_prediction_combination3,
    labels=[0, 1]
).ravel()

print(f"최적 반복 횟수: {model_combination3.best_iteration_}")
print(f"공통 임계값: {COMMON_THRESHOLD:.2f}")
print(f"Precision: {precision_combination3:.6f}")
print(f"Recall: {recall_combination3:.6f}")
print(f"F1-score: {f1_combination3:.6f}")
print(f"ROC-AUC: {roc_auc_combination3:.6f}")
print(f"PR-AUC: {pr_auc_combination3:.6f}")

print("\nConfusion Matrix")
print(f"TN: {tn3:,}")
print(f"FP: {fp3:,}")
print(f"FN: {fn3:,}")
print(f"TP: {tp3:,}")

print("\n예측 결과")
print("실제 이상거래 수:", int(y_val_combination3.sum()))
print("예측 이상거래 수:", int(val_prediction_combination3.sum()))

최적 반복 횟수: 381
공통 임계값: 0.90
Precision: 0.878844
Recall: 0.952032
F1-score: 0.913975
ROC-AUC: 0.999456
PR-AUC: 0.971038

Confusion Matrix
TN: 257,637
FP: 197
FN: 72
TP: 1,429

예측 결과
실제 이상거래 수: 1501
예측 이상거래 수: 1626


조합3은 조합1보다 이상거래를 3건 더 탐지했지만, 오탐은 38건 더 발생. 이 때문에 Recall은 가장 높지만 Precision과 F1-score는 가장 낮음.

현재 F1-score 순위: 
1. 조합1: 0.924174
2. 조합2: 0.916854
3. 조합3: 0.913975

#### 조합 3개가 모두 최적 iteration이 500이 안 되길래, early_stopping = 200으로 재학습해서 성능 유의미하게 좋아지는지 조합1만 먼저 확인하기 위한 실험.

In [38]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

model_combination1_es200 = LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_combination1_es200.fit(
    combination1_data["X_train"],
    combination1_data["y_train"],
    eval_set=[
        (
            combination1_data["X_val"],
            combination1_data["y_val"]
        )
    ],
    eval_metric="auc",
    categorical_feature=combination1_data["categorical_features"],
    callbacks=[
        early_stopping(
            stopping_rounds=200,
            first_metric_only=True,
            verbose=True
        ),
        log_evaluation(period=100)
    ]
);

print("기존 최적 반복:", model_combination1_v2.best_iteration_)
print("기존 최고 ROC-AUC:",
      model_combination1_v2.best_score_["valid_0"]["auc"])

print("\nES 200 최적 반복:",
      model_combination1_es200.best_iteration_)
print("ES 200 최고 ROC-AUC:",
      model_combination1_es200.best_score_["valid_0"]["auc"])

c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.998823
[200]	valid_0's auc: 0.999258
[300]	valid_0's auc: 0.999404
[400]	valid_0's auc: 0.99944
[500]	valid_0's auc: 0.999451
[600]	valid_0's auc: 0.999388
Early stopping, best iteration is:
[492]	valid_0's auc: 0.999456
Evaluated only: auc
기존 최적 반복: 492
기존 최고 ROC-AUC: 0.9994557772807843

ES 200 최적 반복: 492
ES 200 최고 ROC-AUC: 0.9994557772807843


early stopping을 200으로 늘려도 똑같음. 트리를 500개 이상 만들지 않은 게, 학습 부족이 원인이 아니었음. 기존대로 100 ㄱㄱ

## 조합 4 (다영님 모델)

In [ ]:
## 조합4 데이터 준비

combination4_data = prepare_combination_data(
    df=df,
    feature_list=feature_combinations["조합4"],
    train_idx=train_idx,
    val_idx=val_idx,
    target=TARGET
)

print("조합4 변수:")
print(feature_combinations["조합4"])

print("\nTrain 크기:", combination4_data["X_train"].shape)
print("Validation 크기:", combination4_data["X_val"].shape)
print("\n범주형 변수:", combination4_data["categorical_features"])

print(
    "조합1과 행 수 일치:",
    len(combination4_data["X_train"]) == len(combination1_data["X_train"])
    and len(combination4_data["X_val"]) == len(combination1_data["X_val"])
)

조합4 변수:
['recent_24h_high_amt_count', 'amt_to_prior_median_ratio', 'category', 'amt', 'trans_hour', 'age']

Train 크기: (1037340, 6)
Validation 크기: (259335, 6)

범주형 변수: ['category']
조합1과 행 수 일치: True


In [42]:
## 학습

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

model_combination4 = LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_combination4.fit(
    combination4_data["X_train"],
    combination4_data["y_train"],
    eval_set=[
        (
            combination4_data["X_val"],
            combination4_data["y_val"]
        )
    ],
    eval_metric="auc",
    categorical_feature=combination4_data["categorical_features"],
    callbacks=[
        early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
            verbose=True
        ),
        log_evaluation(period=100)
    ]
)

print("최적 반복 횟수:", model_combination4.best_iteration_)
print(
    "최적 Validation ROC-AUC:",
    model_combination4.best_score_["valid_0"]["auc"]
)

c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.998984
[200]	valid_0's auc: 0.999288
[300]	valid_0's auc: 0.999359
[400]	valid_0's auc: 0.999358
Early stopping, best iteration is:
[343]	valid_0's auc: 0.999372
Evaluated only: auc
최적 반복 횟수: 343
최적 Validation ROC-AUC: 0.999372378667718


In [43]:
## validation 예측확률 생성

import numpy as np

val_probability_combination4 = model_combination4.predict_proba(
    combination4_data["X_val"],
    num_iteration=model_combination4.best_iteration_
)[:, 1]

print("예측확률 개수:", len(val_probability_combination4))
print("Validation 행 수:", len(combination4_data["X_val"]))
print("최소 확률:", val_probability_combination4.min())
print("최대 확률:", val_probability_combination4.max())
print("평균 확률:", val_probability_combination4.mean())

print("\n확률 분위수:")
print(
    np.quantile(
        val_probability_combination4,
        [0, 0.5, 0.9, 0.99, 0.995, 0.999, 1.0]
    )
)

예측확률 개수: 259335
Validation 행 수: 259335
최소 확률: 2.608190268090454e-07
최대 확률: 0.9999477841498874
평균 확률: 0.015166390084457025

확률 분위수:
[2.60819027e-07 2.07452103e-04 3.92909549e-03 6.80076986e-01
 9.91777280e-01 9.99845758e-01 9.99947784e-01]


In [44]:
## 공통 임계값 0.90으로 평가

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

COMMON_THRESHOLD = 0.90

y_val_combination4 = combination4_data["y_val"].to_numpy()

val_prediction_combination4 = (
    val_probability_combination4 >= COMMON_THRESHOLD
).astype(int)

precision_combination4 = precision_score(
    y_val_combination4,
    val_prediction_combination4,
    zero_division=0
)

recall_combination4 = recall_score(
    y_val_combination4,
    val_prediction_combination4,
    zero_division=0
)

f1_combination4 = f1_score(
    y_val_combination4,
    val_prediction_combination4,
    zero_division=0
)

roc_auc_combination4 = roc_auc_score(
    y_val_combination4,
    val_probability_combination4
)

pr_auc_combination4 = average_precision_score(
    y_val_combination4,
    val_probability_combination4
)

tn4, fp4, fn4, tp4 = confusion_matrix(
    y_val_combination4,
    val_prediction_combination4,
    labels=[0, 1]
).ravel()

print(f"최적 반복 횟수: {model_combination4.best_iteration_}")
print(f"공통 임계값: {COMMON_THRESHOLD:.2f}")
print(f"Precision: {precision_combination4:.6f}")
print(f"Recall: {recall_combination4:.6f}")
print(f"F1-score: {f1_combination4:.6f}")
print(f"ROC-AUC: {roc_auc_combination4:.6f}")
print(f"PR-AUC: {pr_auc_combination4:.6f}")

print("\nConfusion Matrix")
print(f"TN: {tn4:,}")
print(f"FP: {fp4:,}")
print(f"FN: {fn4:,}")
print(f"TP: {tp4:,}")

print("\n예측 결과")
print("실제 이상거래 수:", int(y_val_combination4.sum()))
print("예측 이상거래 수:", int(val_prediction_combination4.sum()))

최적 반복 횟수: 343
공통 임계값: 0.90
Precision: 0.806215
Recall: 0.950700
F1-score: 0.872516
ROC-AUC: 0.999372
PR-AUC: 0.965424

Confusion Matrix
TN: 257,491
FP: 343
FN: 74
TP: 1,427

예측 결과
실제 이상거래 수: 1501
예측 이상거래 수: 1770


이상거래를 1건 더 탐지, 오탐이 184건 증가.
Precision이 약 9.35%p 감소

현재 F1-score 순위는:

1. 조합1: 0.924174
2. 조합2: 0.916854
3. 조합3: 0.913975
4. 조합4: 0.872516

## 조합 5 (나겸님 모델)

In [45]:
## 조합 5 데이터 준비

combination5_data = prepare_combination_data(
    df=df,
    feature_list=feature_combinations["조합5"],
    train_idx=train_idx,
    val_idx=val_idx,
    target=TARGET
)

print("조합5 변수:")
print(feature_combinations["조합5"])

print("\nTrain 크기:", combination5_data["X_train"].shape)
print("Validation 크기:", combination5_data["X_val"].shape)
print("\n범주형 변수:", combination5_data["categorical_features"])

print(
    "조합1과 행 수 일치:",
    len(combination5_data["X_train"]) == len(combination1_data["X_train"])
    and len(combination5_data["X_val"]) == len(combination1_data["X_val"])
)

조합5 변수:
['category', 'amt', 'is_online', 'recent_24h_high_amt_count', 'category_recent_fraud_rate', 'speed_2', 'customer_mean_amt', 'customer_std_amt', 'amt_ratio_to_mean', 'amt_zscore_card', 'customer_transaction_count', 'trans_hour', 'age', 'rolling_sum_amt_1h', 'prior_normal_median_amt', 'amt_to_prior_median_ratio', 'risk_time_22_04', 'interact_repeat_category', 'has_prior_normal_transaction']

Train 크기: (1037340, 19)
Validation 크기: (259335, 19)

범주형 변수: ['category']
조합1과 행 수 일치: True


In [46]:
from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

model_combination5 = LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_combination5.fit(
    combination5_data["X_train"],
    combination5_data["y_train"],
    eval_set=[
        (
            combination5_data["X_val"],
            combination5_data["y_val"]
        )
    ],
    eval_metric="auc",
    categorical_feature=combination5_data["categorical_features"],
    callbacks=[
        early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
            verbose=True
        ),
        log_evaluation(period=100)
    ]
)

print("최적 반복 횟수:", model_combination5.best_iteration_)
print(
    "최적 Validation ROC-AUC:",
    model_combination5.best_score_["valid_0"]["auc"]
)

c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.998278
[200]	valid_0's auc: 0.999048
[300]	valid_0's auc: 0.999314
[400]	valid_0's auc: 0.999459
[500]	valid_0's auc: 0.999545
[600]	valid_0's auc: 0.999602
[700]	valid_0's auc: 0.999621
[800]	valid_0's auc: 0.999637
[900]	valid_0's auc: 0.999654
[1000]	valid_0's auc: 0.999671
[1100]	valid_0's auc: 0.999688
[1200]	valid_0's auc: 0.999703
[1300]	valid_0's auc: 0.999719
[1400]	valid_0's auc: 0.999731
[1500]	valid_0's auc: 0.999744
[1600]	valid_0's auc: 0.999748
[1700]	valid_0's auc: 0.99975
Early stopping, best iteration is:
[1680]	valid_0's auc: 0.999751
Evaluated only: auc
최적 반복 횟수: 1680
최적 Validation ROC-AUC: 0.9997505250745775


In [47]:
import numpy as np

val_probability_combination5 = model_combination5.predict_proba(
    combination5_data["X_val"],
    num_iteration=model_combination5.best_iteration_
)[:, 1]

print("예측확률 개수:", len(val_probability_combination5))
print("Validation 행 수:", len(combination5_data["X_val"]))
print("최소 확률:", val_probability_combination5.min())
print("최대 확률:", val_probability_combination5.max())
print("평균 확률:", val_probability_combination5.mean())

print("\n확률 분위수:")
print(
    np.quantile(
        val_probability_combination5,
        [0, 0.5, 0.9, 0.99, 0.995, 0.999, 1.0]
    )
)

예측확률 개수: 259335
Validation 행 수: 259335
최소 확률: 2.1003004960688718e-13
최대 확률: 0.9999999997525739
평균 확률: 0.006356377687915276

확률 분위수:
[2.10030050e-13 2.35975817e-07 3.54820289e-05 1.76913952e-02
 9.96464774e-01 9.99999697e-01 1.00000000e+00]


In [48]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

COMMON_THRESHOLD = 0.90

y_val_combination5 = combination5_data["y_val"].to_numpy()

val_prediction_combination5 = (
    val_probability_combination5 >= COMMON_THRESHOLD
).astype(int)

precision_combination5 = precision_score(
    y_val_combination5,
    val_prediction_combination5,
    zero_division=0
)

recall_combination5 = recall_score(
    y_val_combination5,
    val_prediction_combination5,
    zero_division=0
)

f1_combination5 = f1_score(
    y_val_combination5,
    val_prediction_combination5,
    zero_division=0
)

roc_auc_combination5 = roc_auc_score(
    y_val_combination5,
    val_probability_combination5
)

pr_auc_combination5 = average_precision_score(
    y_val_combination5,
    val_probability_combination5
)

tn5, fp5, fn5, tp5 = confusion_matrix(
    y_val_combination5,
    val_prediction_combination5,
    labels=[0, 1]
).ravel()

print(f"최적 반복 횟수: {model_combination5.best_iteration_}")
print(f"공통 임계값: {COMMON_THRESHOLD:.2f}")
print(f"Precision: {precision_combination5:.6f}")
print(f"Recall: {recall_combination5:.6f}")
print(f"F1-score: {f1_combination5:.6f}")
print(f"ROC-AUC: {roc_auc_combination5:.6f}")
print(f"PR-AUC: {pr_auc_combination5:.6f}")

print("\nConfusion Matrix")
print(f"TN: {tn5:,}")
print(f"FP: {fp5:,}")
print(f"FN: {fn5:,}")
print(f"TP: {tp5:,}")

print("\n예측 결과")
print("실제 이상거래 수:", int(y_val_combination5.sum()))
print("예측 이상거래 수:", int(val_prediction_combination5.sum()))

최적 반복 횟수: 1680
공통 임계값: 0.90
Precision: 0.968793
Recall: 0.930713
F1-score: 0.949371
ROC-AUC: 0.999751
PR-AUC: 0.982701

Confusion Matrix
TN: 257,789
FP: 45
FN: 104
TP: 1,397

예측 결과
실제 이상거래 수: 1501
예측 이상거래 수: 1442


현재까지 1위였던 조합 1과 비교하면,

조합 5는
- 오탐(FP) 114건 감소 -> Precision 약 6.91%p 상승
- 미탐(FN) 29건 증가 -> Recall 약 1.93%p 하락

조합5는 정상거래를 이상거래로 잘못 탐지하는 문제를 크게 줄인 대신, 이상거래를 조금 더 놓치는 결과.

### 최종 F1 순위는:

1. 조합5: 0.949371 <- 선택.
2. 조합1: 0.924174
3. 조합2: 0.916854
4. 조합3: 0.913975
5. 조합4: 0.872516

## 2. 시간순 교차검증 및 하이퍼파라미터 튜닝

시간순 Fold를 나누기 전에 데이터의 시간 범위와 월별 이상거래 분포부터 확인

In [ ]:
### 월별 거래 건수·이상거래 수·이상거래율 확인 코드

import pandas as pd

TIME_COL = "trans_date_trans_time"
TARGET = "is_fraud"

# 시간형으로 변환
df[TIME_COL] = pd.to_datetime(df[TIME_COL])

# 전체 데이터를 거래 발생 시간순으로 정렬
df = (
    df.sort_values(TIME_COL)
      .reset_index(drop=True)
)

print("전체 행 수:", len(df))
print("시작 시점:", df[TIME_COL].min())
print("종료 시점:", df[TIME_COL].max())
print("전체 이상거래 수:", int(df[TARGET].sum()))
print("전체 이상거래율:", df[TARGET].mean())

# 월별 거래 및 이상거래 분포
monthly_distribution = (
    df.assign(month=df[TIME_COL].dt.to_period("M"))
      .groupby("month", observed=True)
      .agg(
          transaction_count=(TARGET, "size"),
          fraud_count=(TARGET, "sum"),
          fraud_rate=(TARGET, "mean")
      )
      .reset_index()
)

monthly_distribution["fraud_rate_pct"] = (
    monthly_distribution["fraud_rate"] * 100
)

print("\n월별 분포")
display(
    monthly_distribution[
        [
            "month",
            "transaction_count",
            "fraud_count",
            "fraud_rate_pct"
        ]
    ]
)

전체 행 수: 1296675
시작 시점: 2019-01-01 00:00:18
종료 시점: 2020-06-21 12:13:37
전체 이상거래 수: 7506
전체 이상거래율: 0.005788651743883394

월별 분포


,month,transaction_count,fraud_count,fraud_rate_pct
0,2019-01,52525,506,0.963351
1,2019-02,49866,517,1.036779
2,2019-03,70939,494,0.696373
3,2019-04,68078,376,0.552308
4,2019-05,72532,408,0.562510
5,2019-06,86064,354,0.411322
6,2019-07,86596,331,0.382235
7,2019-08,87359,382,0.437276
8,2019-09,70652,418,0.591632
9,2019-10,68758,454,0.660287


월별 분포를 보면 모든 달에 이상거래가 300건 이상 있어서, 시간순 검증 구간에 이상거래가 부족할 가능성은 낮음.

 따라서 노션에 적어놓은대로 행 수 기준 3-Fold Expanding Window를 적용.

In [50]:
## Fold 경계만 생성하고 기간/이상거래 분포 확인.
import numpy as np
import pandas as pd

TIME_COL = "trans_date_trans_time"
TARGET = "is_fraud"

# 시간순 정렬 상태 재확인
df[TIME_COL] = pd.to_datetime(df[TIME_COL])

df = (
    df.sort_values(TIME_COL)
      .reset_index(drop=True)
)

n = len(df)

# 전체 데이터를 6개 구간으로 나누기 위한 경계
boundaries = np.linspace(0, n, 7, dtype=int)

time_folds = []

for fold_number in range(1, 4):

    train_end = boundaries[fold_number + 2]
    val_start = train_end
    val_end = boundaries[fold_number + 3]

    train_idx_fold = np.arange(0, train_end)
    val_idx_fold = np.arange(val_start, val_end)

    time_folds.append(
        {
            "fold": fold_number,
            "train_idx": train_idx_fold,
            "val_idx": val_idx_fold
        }
    )

    train_part = df.iloc[train_idx_fold]
    val_part = df.iloc[val_idx_fold]

    print(f"\n===== Fold {fold_number} =====")

    print(
        "Train 기간:",
        train_part[TIME_COL].min(),
        "~",
        train_part[TIME_COL].max()
    )
    print("Train 행 수:", len(train_part))
    print("Train 이상거래 수:", int(train_part[TARGET].sum()))
    print("Train 이상거래율:", train_part[TARGET].mean())

    print(
        "\nValidation 기간:",
        val_part[TIME_COL].min(),
        "~",
        val_part[TIME_COL].max()
    )
    print("Validation 행 수:", len(val_part))
    print("Validation 이상거래 수:", int(val_part[TARGET].sum()))
    print("Validation 이상거래율:", val_part[TARGET].mean())

    print(
        "\n시간 순서 정상:",
        train_part[TIME_COL].max()
        <= val_part[TIME_COL].min()
    )


===== Fold 1 =====
Train 기간: 2019-01-01 00:00:18 ~ 2019-10-03 07:35:11
Train 행 수: 648337
Train 이상거래 수: 3827
Train 이상거래율: 0.005902794380083198

Validation 기간: 2019-10-03 07:35:47 ~ 2019-12-18 17:06:47
Validation 행 수: 216113
Validation 이상거래 수: 1091
Validation 이상거래율: 0.0050482849250160795

시간 순서 정상: True

===== Fold 2 =====
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-18 17:06:47
Train 행 수: 864450
Train 이상거래 수: 4918
Train 이상거래율: 0.005689166522066053

Validation 기간: 2019-12-18 17:07:05 ~ 2020-03-24 15:14:25
Validation 행 수: 216112
Validation 이상거래 수: 1363
Validation 이상거래율: 0.0063069149329977045

시간 순서 정상: True

===== Fold 3 =====
Train 기간: 2019-01-01 00:00:18 ~ 2020-03-24 15:14:25
Train 행 수: 1080562
Train 이상거래 수: 6281
Train 이상거래율: 0.005812715975575673

Validation 기간: 2020-03-24 15:14:30 ~ 2020-06-21 12:13:37
Validation 행 수: 216113
Validation 이상거래 수: 1225
Validation 이상거래율: 0.005668330919472683

시간 순서 정상: True


In [ ]:
## 조합 1 Fold별 데이터 준비

combination1_timefold_data = []

for fold_info in time_folds:
    
    fold_number = fold_info["fold"]
    
    fold_data = prepare_combination_data(
        df=df,
        feature_list=feature_combinations["조합1"],
        train_idx=fold_info["train_idx"],
        val_idx=fold_info["val_idx"],
        target=TARGET
    )
    
    combination1_timefold_data.append(
        {
            "fold": fold_number,
            **fold_data
        }
    )
    
    print(f"\n===== 조합1 Fold {fold_number} =====")
    print("Train 크기:", fold_data["X_train"].shape)
    print("Validation 크기:", fold_data["X_val"].shape)
    print("Train 이상거래 수:", int(fold_data["y_train"].sum()))
    print("Validation 이상거래 수:", int(fold_data["y_val"].sum()))
    print("범주형 변수:", fold_data["categorical_features"])


===== 조합1 Fold 1 =====
Train 크기: (648337, 10)
Validation 크기: (216113, 10)
Train 이상거래 수: 3827
Validation 이상거래 수: 1091
범주형 변수: ['category']

===== 조합1 Fold 2 =====
Train 크기: (864450, 10)
Validation 크기: (216112, 10)
Train 이상거래 수: 4918
Validation 이상거래 수: 1363
범주형 변수: ['category']

===== 조합1 Fold 3 =====
Train 크기: (1080562, 10)
Validation 크기: (216113, 10)
Train 이상거래 수: 6281
Validation 이상거래 수: 1225
범주형 변수: ['category']


In [52]:
## 조합 5의 Fold별 데이터 준비

combination5_timefold_data = []

for fold_info in time_folds:
    
    fold_number = fold_info["fold"]
    
    fold_data = prepare_combination_data(
        df=df,
        feature_list=feature_combinations["조합5"],
        train_idx=fold_info["train_idx"],
        val_idx=fold_info["val_idx"],
        target=TARGET
    )
    
    combination5_timefold_data.append(
        {
            "fold": fold_number,
            **fold_data
        }
    )
    
    print(f"\n===== 조합5 Fold {fold_number} =====")
    print("Train 크기:", fold_data["X_train"].shape)
    print("Validation 크기:", fold_data["X_val"].shape)
    print("Train 이상거래 수:", int(fold_data["y_train"].sum()))
    print("Validation 이상거래 수:", int(fold_data["y_val"].sum()))
    print("범주형 변수:", fold_data["categorical_features"])


===== 조합5 Fold 1 =====
Train 크기: (648337, 19)
Validation 크기: (216113, 19)
Train 이상거래 수: 3827
Validation 이상거래 수: 1091
범주형 변수: ['category']

===== 조합5 Fold 2 =====
Train 크기: (864450, 19)
Validation 크기: (216112, 19)
Train 이상거래 수: 4918
Validation 이상거래 수: 1363
범주형 변수: ['category']

===== 조합5 Fold 3 =====
Train 크기: (1080562, 19)
Validation 크기: (216113, 19)
Train 이상거래 수: 6281
Validation 이상거래 수: 1225
범주형 변수: ['category']


하이퍼파라미터 튜닝의 1차 기준은 불균형 데이터에 적합한 3개 Fold의 평균 PR-AUC.

- 1차 기준: 평균 PR-AUC가 높은 설정
- 2차 기준: Fold별 PR-AUC 표준편차가 작은 설정

참고 지표: ROC-AUC

F1-score: 아직 사용하지 않음
→ 임계값에 영향을 받기 때문에 최종 모델 선정 후 별도로 튜닝

In [54]:
## 조합1의 기본 설정 시간순 교차검증
## scale_pos_weight도 각 Fold의 학습 데이터만으로 다시 계산해 미래 정보가 들어가지 않게함.

import numpy as np
import pandas as pd

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

combination1_baseline_results = []

for fold_data in combination1_timefold_data:

    fold_number = fold_data["fold"]

    X_train = fold_data["X_train"]
    y_train = fold_data["y_train"]
    X_val = fold_data["X_val"]
    y_val = fold_data["y_val"]

    # Fold별 학습 데이터에서만 불균형 비율 계산
    fold_scale_pos_weight = (
        (y_train == 0).sum() / (y_train == 1).sum()
    )

    model = LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=31,
        scale_pos_weight=fold_scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="average_precision",
        categorical_feature=fold_data["categorical_features"],
        callbacks=[
            early_stopping(
                stopping_rounds=100,
                first_metric_only=True,
                verbose=False
            ),
            log_evaluation(period=0)
        ]
    )

    val_probability = model.predict_proba(
        X_val,
        num_iteration=model.best_iteration_
    )[:, 1]

    fold_pr_auc = average_precision_score(
        y_val,
        val_probability
    )

    fold_roc_auc = roc_auc_score(
        y_val,
        val_probability
    )

    combination1_baseline_results.append(
        {
            "fold": fold_number,
            "best_iteration": model.best_iteration_,
            "scale_pos_weight": fold_scale_pos_weight,
            "pr_auc": fold_pr_auc,
            "roc_auc": fold_roc_auc
        }
    )

    print(
        f"Fold {fold_number} 완료 | "
        f"Best iteration: {model.best_iteration_} | "
        f"PR-AUC: {fold_pr_auc:.6f} | "
        f"ROC-AUC: {fold_roc_auc:.6f}"
    )

combination1_baseline_df = pd.DataFrame(
    combination1_baseline_results
)

print("\n조합1 기본 설정 결과")
display(combination1_baseline_df)

print(
    f"평균 PR-AUC: "
    f"{combination1_baseline_df['pr_auc'].mean():.6f}"
)
print(
    f"PR-AUC 표준편차: "
    f"{combination1_baseline_df['pr_auc'].std(ddof=1):.6f}"
)
print(
    f"평균 ROC-AUC: "
    f"{combination1_baseline_df['roc_auc'].mean():.6f}"
)

c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1 완료 | Best iteration: 613 | PR-AUC: 0.966643 | ROC-AUC: 0.999171


c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2 완료 | Best iteration: 1185 | PR-AUC: 0.977094 | ROC-AUC: 0.999203


c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 3 완료 | Best iteration: 713 | PR-AUC: 0.974724 | ROC-AUC: 0.999511

조합1 기본 설정 결과


,fold,best_iteration,scale_pos_weight,pr_auc,roc_auc
0,1,613,168.411288,0.966643,0.999171
1,2,1185,174.772672,0.977094,0.999203
2,3,713,171.036618,0.974724,0.999511


평균 PR-AUC: 0.972820
PR-AUC 표준편차: 0.005479
평균 ROC-AUC: 0.999295


Fold1이 조금 낮지만, 편차가 크지않아, 시간 구간이 바뀌어도 비교적 안정적인 성능.

In [55]:
## 동일한 기본 설정으로, 조합 5만 시간순 교차검증.

combination5_baseline_results = []

for fold_data in combination5_timefold_data:

    fold_number = fold_data["fold"]

    X_train = fold_data["X_train"]
    y_train = fold_data["y_train"]
    X_val = fold_data["X_val"]
    y_val = fold_data["y_val"]

    # 각 Fold의 학습 데이터만으로 계산
    fold_scale_pos_weight = (
        (y_train == 0).sum() / (y_train == 1).sum()
    )

    model = LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=31,
        scale_pos_weight=fold_scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train,
        y_train,
        eval_X=X_val,
        eval_y=y_val,
        eval_metric="average_precision",
        categorical_feature=fold_data["categorical_features"],
        callbacks=[
            early_stopping(
                stopping_rounds=100,
                first_metric_only=True,
                verbose=False
            ),
            log_evaluation(period=0)
        ]
    )

    val_probability = model.predict_proba(
        X_val,
        num_iteration=model.best_iteration_
    )[:, 1]

    fold_pr_auc = average_precision_score(
        y_val,
        val_probability
    )

    fold_roc_auc = roc_auc_score(
        y_val,
        val_probability
    )

    combination5_baseline_results.append(
        {
            "fold": fold_number,
            "best_iteration": model.best_iteration_,
            "scale_pos_weight": fold_scale_pos_weight,
            "pr_auc": fold_pr_auc,
            "roc_auc": fold_roc_auc
        }
    )

    print(
        f"Fold {fold_number} 완료 | "
        f"Best iteration: {model.best_iteration_} | "
        f"PR-AUC: {fold_pr_auc:.6f} | "
        f"ROC-AUC: {fold_roc_auc:.6f}"
    )

combination5_baseline_df = pd.DataFrame(
    combination5_baseline_results
)

print("\n조합5 기본 설정 결과")
display(combination5_baseline_df)

print(
    f"평균 PR-AUC: "
    f"{combination5_baseline_df['pr_auc'].mean():.6f}"
)
print(
    f"PR-AUC 표준편차: "
    f"{combination5_baseline_df['pr_auc'].std(ddof=1):.6f}"
)
print(
    f"평균 ROC-AUC: "
    f"{combination5_baseline_df['roc_auc'].mean():.6f}"
)

Fold 1 완료 | Best iteration: 600 | PR-AUC: 0.961376 | ROC-AUC: 0.998033
Fold 2 완료 | Best iteration: 231 | PR-AUC: 0.961957 | ROC-AUC: 0.998903
Fold 3 완료 | Best iteration: 865 | PR-AUC: 0.972417 | ROC-AUC: 0.999485

조합5 기본 설정 결과


,fold,best_iteration,scale_pos_weight,pr_auc,roc_auc
0,1,600,168.411288,0.961376,0.998033
1,2,231,174.772672,0.961957,0.998903
2,3,865,171.036618,0.972417,0.999485


평균 PR-AUC: 0.965250
PR-AUC 표준편차: 0.006214
평균 ROC-AUC: 0.998807


## 시간순 교차검증에서는, 조합1이 조합5보다 성능과 안정성 모두 조금 더 우수.

조합5는 기존 무작위 Validation에서는 가장 좋았지만, 시간순 교차검증에서는 조합1보다 낮아짐. 추가 변수들이 특정 시기에서는 강했지만 미래 구간으로 넘어가면서 관계가 조금 변했을 가능성이 있음. 다만 아직 기본 설정 결과이므로, 계획대로 두 조합 모두 같은 범위에서 튜닝한 뒤 최종 판단해야 함.

그리고 조합3도 추가. 총 3개 모델(조합1(민정40), 조합3(민정44), 조합5(나겸)) 비교.

In [ ]:
## 조합3의 Fold별 데이터 준비

combination3_timefold_data = []

for fold_info in time_folds:
    
    fold_number = fold_info["fold"]
    
    fold_data = prepare_combination_data(
        df=df,
        feature_list=feature_combinations["조합3"],
        train_idx=fold_info["train_idx"],
        val_idx=fold_info["val_idx"],
        target=TARGET
    )
    
    combination3_timefold_data.append(
        {
            "fold": fold_number,
            **fold_data
        }
    )
    
    print(f"\n===== 조합3 Fold {fold_number} =====")
    print("Train 크기:", fold_data["X_train"].shape)
    print("Validation 크기:", fold_data["X_val"].shape)
    print("Train 이상거래 수:", int(fold_data["y_train"].sum()))
    print("Validation 이상거래 수:", int(fold_data["y_val"].sum()))
    print("범주형 변수:", fold_data["categorical_features"])


===== 조합3 Fold 1 =====
Train 크기: (648337, 11)
Validation 크기: (216113, 11)
Train 이상거래 수: 3827
Validation 이상거래 수: 1091
범주형 변수: ['category']

===== 조합3 Fold 2 =====
Train 크기: (864450, 11)
Validation 크기: (216112, 11)
Train 이상거래 수: 4918
Validation 이상거래 수: 1363
범주형 변수: ['category']

===== 조합3 Fold 3 =====
Train 크기: (1080562, 11)
Validation 크기: (216113, 11)
Train 이상거래 수: 6281
Validation 이상거래 수: 1225
범주형 변수: ['category']


In [57]:
## 조합3 시간순 3-Fold 교차검증

combination3_baseline_results = []

for fold_data in combination3_timefold_data:

    fold_number = fold_data["fold"]

    X_train = fold_data["X_train"]
    y_train = fold_data["y_train"]
    X_val = fold_data["X_val"]
    y_val = fold_data["y_val"]

    # 해당 Fold의 학습 데이터만 이용해 불균형 비율 계산
    fold_scale_pos_weight = (
        (y_train == 0).sum() / (y_train == 1).sum()
    )

    model = LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=20,
        scale_pos_weight=fold_scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train,
        y_train,
        eval_X=X_val,
        eval_y=y_val,
        eval_metric="average_precision",
        categorical_feature=fold_data["categorical_features"],
        callbacks=[
            early_stopping(
                stopping_rounds=100,
                first_metric_only=True,
                verbose=False
            ),
            log_evaluation(period=0)
        ]
    )

    val_probability = model.predict_proba(
        X_val,
        num_iteration=model.best_iteration_
    )[:, 1]

    fold_pr_auc = average_precision_score(
        y_val,
        val_probability
    )

    fold_roc_auc = roc_auc_score(
        y_val,
        val_probability
    )

    combination3_baseline_results.append(
        {
            "fold": fold_number,
            "best_iteration": model.best_iteration_,
            "scale_pos_weight": fold_scale_pos_weight,
            "pr_auc": fold_pr_auc,
            "roc_auc": fold_roc_auc
        }
    )

    print(
        f"Fold {fold_number} 완료 | "
        f"Best iteration: {model.best_iteration_} | "
        f"PR-AUC: {fold_pr_auc:.6f} | "
        f"ROC-AUC: {fold_roc_auc:.6f}"
    )

combination3_baseline_df = pd.DataFrame(
    combination3_baseline_results
)

print("\n조합3 기본 설정 결과")
display(combination3_baseline_df)

print(
    f"평균 PR-AUC: "
    f"{combination3_baseline_df['pr_auc'].mean():.6f}"
)
print(
    f"PR-AUC 표준편차: "
    f"{combination3_baseline_df['pr_auc'].std(ddof=1):.6f}"
)
print(
    f"평균 ROC-AUC: "
    f"{combination3_baseline_df['roc_auc'].mean():.6f}"
)

Fold 1 완료 | Best iteration: 1486 | PR-AUC: 0.968004 | ROC-AUC: 0.999027
Fold 2 완료 | Best iteration: 695 | PR-AUC: 0.976293 | ROC-AUC: 0.999357
Fold 3 완료 | Best iteration: 881 | PR-AUC: 0.975818 | ROC-AUC: 0.999549

조합3 기본 설정 결과


,fold,best_iteration,scale_pos_weight,pr_auc,roc_auc
0,1,1486,168.411288,0.968004,0.999027
1,2,695,174.772672,0.976293,0.999357
2,3,881,171.036618,0.975818,0.999549


평균 PR-AUC: 0.973372
PR-AUC 표준편차: 0.004655
평균 ROC-AUC: 0.999311


기본 설정 기준으로는 조합3이 세 조합 중 가장 우수함.

## 각 조합마다 개별적으로 튜닝하기